In [7]:
from collections import Counter

splits = {
    "cat": ["c", "##a", "##t"],
    "cats": ["c", "##a", "##t", "##s"],
    "cater": ["c", "##a", "##t", "##e", "##r"],
    "dog": ["d", "##o", "##g"]
}

word_freq = {
    "cat": 3,
    "cats": 2,
    "cater": 1,
    "dog": 2
}


token_freq = Counter()
pair_freq = Counter()

for word, freq in word_freq.items():

    tokens = splits[word]

    for token in tokens:
        token_freq[token] += freq

    for i in range(len(tokens) - 1):
        pair = (tokens[i], tokens[i + 1])
        pair_freq[pair] += freq

print("Token Frequencies:")
print(token_freq)

print("\nPair Frequencies:")
print(pair_freq)

scores = {}

for pair, freq in pair_freq.items():

    first = pair[0]
    second = pair[1]

    score = freq / (
        token_freq[first] * token_freq[second]
    )

    scores[pair] = score

print("\nWordPiece Scores:")

for pair, score in scores.items():
    print(pair, "=", round(score, 4))


best_pair = max(scores, key=scores.get)

print("\nBest Pair:", best_pair)

def merge_pair(splits, pair):

    new_token = pair[0] + pair[1].replace("##", "")

    for word in splits:

        tokens = splits[word]
        new_tokens = []

        i = 0

        while i < len(tokens):

            if i < len(tokens) - 1:

                current_pair = (
                    tokens[i],
                    tokens[i + 1]
                )

                if current_pair == pair:

                    new_tokens.append(new_token)
                    i += 2
                    continue

            new_tokens.append(tokens[i])
            i += 1

        splits[word] = new_tokens

    return new_token


new_token = merge_pair(splits, best_pair)

print("\nNew Token:", new_token)

print("\nUpdated Splits:")
print(splits)


vocab = {
    "c",
    "d",
    "##a",
    "##t",
    "##s",
    "##e",
    "##r",
    "##o",
    "##g",
    "ca",
    "cat",
    "##er"
}


def tokenize_word(word, vocab):

    tokens = []

    while len(word) > 0:

        found = False

        for i in range(len(word), 0, -1):

            part = word[:i]

            if len(tokens) > 0:
                part = "##" + part

            if part in vocab:

                tokens.append(part)
                word = word[i:]
                found = True
                break

        if not found:
            return ["[UNK]"]

    return tokens


# Test
word = "cats"

tokens = tokenize_word(word, vocab)

print("\nInput Word:", word)
print("Tokens:", tokens)

vocab_ids = {
    "[UNK]": 0,
    "c": 1,
    "d": 2,
    "##a": 3,
    "##t": 4,
    "##s": 5,
    "##e": 6,
    "##r": 7,
    "##o": 8,
    "##g": 9,
    "ca": 10,
    "cat": 11,
    "##er": 12
}

ids = [vocab_ids[token] for token in tokens]

print("\nTokens:", tokens)
print("Token IDs:", ids)

Token Frequencies:
Counter({'c': 6, '##a': 6, '##t': 6, '##s': 2, 'd': 2, '##o': 2, '##g': 2, '##e': 1, '##r': 1})

Pair Frequencies:
Counter({('c', '##a'): 6, ('##a', '##t'): 6, ('##t', '##s'): 2, ('d', '##o'): 2, ('##o', '##g'): 2, ('##t', '##e'): 1, ('##e', '##r'): 1})

WordPiece Scores:
('c', '##a') = 0.1667
('##a', '##t') = 0.1667
('##t', '##s') = 0.1667
('##t', '##e') = 0.1667
('##e', '##r') = 1.0
('d', '##o') = 0.5
('##o', '##g') = 0.5

Best Pair: ('##e', '##r')

New Token: ##er

Updated Splits:
{'cat': ['c', '##a', '##t'], 'cats': ['c', '##a', '##t', '##s'], 'cater': ['c', '##a', '##t', '##er'], 'dog': ['d', '##o', '##g']}

Input Word: cats
Tokens: ['cat', '##s']

Tokens: ['cat', '##s']
Token IDs: [11, 5]
